# Laboratorio Semana 6: Preparación de Datos y Benchmarking (CSV vs Parquet)
**Materia:** Fundamentos de Big Data (IDSD413) - Escuela Politécnica Nacional
**Profesor:** David Esteban Galarza García

En este laboratorio demostraremos empíricamente el problema del cuello de botella de I/O (Lectura/Escritura) en Big Data. Generaremos un dataset sintético masivo de transacciones de retail y compararemos el rendimiento de un formato basado en filas (CSV) frente a un formato columnar nativo de Big Data (Apache Parquet).

In [ ]:
import pandas as pd
import numpy as np
import time
import os

# Reto 1: Se aumenta de 5M a 15M de filas para probar el consumo de RAM
N_ROWS = 15_000_000

print(f"Generando {N_ROWS} registros en memoria RAM...")
start_time = time.time()

# Generación de datos de prueba para transacciones
np.random.seed(42)
df = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

print(f"Dataset generado en {time.time() - start_time:.2f} segundos.")
# Revisamos cuánto pesa el DataFrame completo en RAM antes de pasarlo a disco
df.info(memory_usage='deep')

## 1. Prueba de Escritura y Compresión en Disco
Guardaremos exactamente el mismo DataFrame en ambos formatos para analizar los tiempos de escritura y la eficiencia del algoritmo de compresión (Snappy) que utiliza Parquet por defecto.

In [ ]:
# 1.1 Escritura en CSV (Basado en filas, texto plano)
print("Escribiendo archivo CSV...")
start_csv_write = time.time()
df.to_csv('transacciones_epn.csv', index=False)
csv_write_time = time.time() - start_csv_write
print(f"Tiempo de escritura CSV: {csv_write_time:.2f} segundos")

# 1.2 Escritura en Parquet (Columnar, tipado fuerte, comprimido)
print("\nEscribiendo archivo Parquet...")
start_parquet_write = time.time()
df.to_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_write_time = time.time() - start_parquet_write
print(f"Tiempo de escritura Parquet: {parquet_write_time:.2f} segundos")

# 1.3 Comparación de peso en el almacenamiento
csv_size = os.path.getsize('transacciones_epn.csv') / (1024 * 1024)
parquet_size = os.path.getsize('transacciones_epn.parquet') / (1024 * 1024)

print("-" * 40)
print(f"Peso del CSV: {csv_size:.2f} MB")
print(f"Peso del Parquet: {parquet_size:.2f} MB")
print(f"Reducción de almacenamiento: {((csv_size - parquet_size) / csv_size) * 100:.2f}%")

## 2. Prueba de Lectura Completa (Full Scan)
Evaluaremos cuánto tarda el procesador en levantar la totalidad del archivo desde el disco duro hacia la memoria RAM de Google Colab.

In [ ]:
# Limpiamos de la RAM el DataFrame inicial, para que no se dupliquen los 15M de registros
del df
import gc
gc.collect()

# 2.1 Lectura de CSV (Full Scan)
print("Leyendo CSV a memoria...")
start_csv_read = time.time()
df_csv = pd.read_csv('transacciones_epn.csv')
csv_read_time = time.time() - start_csv_read

# Calculamos cuánta memoria RAM(mb) ocupa el DataFrame cargado desde el CSV
memoria_csv = df_csv.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Tiempo de lectura CSV: {csv_read_time:.2f} segundos | Uso en RAM: {memoria_csv:.2f} MB")

# Borramos df_csv de la memoria antes de la siguiente prueba para no duplicar los 15M de registros en RAM
del df_csv
gc.collect()

# 2.2 Lectura de Parquet (Full Scan)
print("\nLeyendo Parquet a memoria...")
start_parquet_read = time.time()

# PyArrow lee los bloques columnares comprimidos y ya conoce el tipo de dato exacto de cada columna
df_parquet = pd.read_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_read_time = time.time() - start_parquet_read

# Calculamos cuánta memoria RAM (en MB) ocupa el DataFrame cargado desde Parquet
memoria_parquet = df_parquet.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Tiempo de lectura Parquet: {parquet_read_time:.2f} segundos | Uso en RAM: {memoria_parquet:.2f} MB")

# Liberamos la memoria para dejar el entorno limpio antes de pasar a las consultas selectivas
del df_parquet
gc.collect()

## 3. Ventaja Columnar: Consultas Selectivas (Column Pruning)
En Ciencia de Datos, rara vez analizamos todas las columnas a la vez. En este escenario, el equipo de analítica solo necesita calcular el total de ventas (columna `monto`).

Veremos cómo Parquet escanea *únicamente* los bytes de esa columna, mientras que CSV está obligado a leer el archivo entero.

In [ ]:
# 3.1 Agregación selectiva en CSV
print("Calculando ventas totales desde CSV...")
start_csv_query = time.time()
# Pandas tiene que analizar todo el archivo para extraer una sola columna
suma_csv = pd.read_csv('transacciones_epn.csv', usecols=['monto'])['monto'].sum()
csv_query_time = time.time() - start_csv_query
print(f"Ventas (CSV): ${suma_csv:,.2f} | Tiempo: {csv_query_time:.4f} segundos")

# 3.2 Agregación selectiva en Parquet
print("\nCalculando ventas totales desde Parquet...")
start_parquet_query = time.time()
# PyArrow inyecta la proyección directamente a nivel de disco
suma_parquet = pd.read_parquet('transacciones_epn.parquet', columns=['monto'], engine='pyarrow')['monto'].sum()
parquet_query_time = time.time() - start_parquet_query
print(f"Ventas (Parquet): ${suma_parquet:,.2f} | Tiempo: {parquet_query_time:.4f} segundos")

print("-" * 40)
print(f"Aceleración de consulta selectiva: {csv_query_time / parquet_query_time:.2f} veces más rápido")

## 4. Reto en Vivo 1: Filtrado desde el Disco (Predicate Pushdown)
¿Qué pasa si nuestro modelo solo necesita los datos de 'Electronica'?
En CSV, la memoria RAM puede colapsar porque Pandas debe cargar todo el archivo antes de filtrar. En Parquet, el motor filtra los datos directamente en el disco duro y solo envía a la RAM lo que coincide.

In [ ]:
# 1. El método tradicional (CSV)
print("Filtrando CSV (Carga total -> Filtro)...")
start_csv = time.time()
df_temp_csv = pd.read_csv('transacciones_epn.csv')
df_filtrado_csv = df_temp_csv[(df_temp_csv['categoria'] == 'Juguetes') & (df_temp_csv['estado'] == 'Completado')]['monto']
tiempo_csv = time.time() - start_csv
print(f"Tiempo CSV: {tiempo_csv:.4f} segundos | Registros: {len(df_filtrado_csv)}")

# Borramos las variables temporales y llamamos al recolector de basura para liberar la RAM
del df_temp_csv, df_filtrado_csv
import gc; gc.collect()

# 2. El método Big Data (Parquet con Predicate Pushdown y Column Pruning)
print("\nFiltrando Parquet (Filtro en disco -> Carga parcial)...")
start_parquet = time.time()

# Con PyArrow aplicamos las dos optimizaciones directamente en la lectura:
# - columns=['monto']: aplica Column Pruning para leer solo esa columna
# - filters=[...]: aplica Predicate Pushdown evaluando cada tupla
df_filtrado_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    engine='pyarrow',
    columns=['monto'],
    filters=[
        ('categoria', '=', 'Juguetes'),
        ('estado', '=', 'Completado')
    ]
)

tiempo_parquet = time.time() - start_parquet
print(f"Tiempo Parquet: {tiempo_parquet:.4f} segundos | Registros: {len(df_filtrado_parquet)}")

print("-" * 40)
print(f"Aceleración: {tiempo_csv / tiempo_parquet:.2f}x más rápido")

## 5. Reto en Vivo 2: Agrupación y Estadísticas (Group By)
El análisis exploratorio requiere agrupar datos. Veamos cómo impacta la lectura columnar al calcular el promedio de ventas agrupado por el estado de la transacción.

In [ ]:
# 1. Agrupación en CSV
print("Agrupando desde CSV...")
start_csv_grp = time.time()
grupo_csv = pd.read_csv('transacciones_epn.csv').groupby('estado')['monto'].mean()
tiempo_csv_grp = time.time() - start_csv_grp
print(f"Tiempo CSV: {tiempo_csv_grp:.4f} segundos")

# 2. Agrupación en Parquet
print("\nAgrupando desde Parquet...")
start_parquet_grp = time.time()
grupo_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    columns=['estado', 'monto'],
    engine='pyarrow'
).groupby('estado')['monto'].mean()
tiempo_parquet_grp = time.time() - start_parquet_grp
print(f"Tiempo Parquet: {tiempo_parquet_grp:.4f} segundos")

print("-" * 40)
print("Resultados de la agregación:")
print(grupo_parquet)